# W22 · rclpy：用 Python 写 ROS2 节点

> 上一讲你用 CLI 操控别人的节点；这一讲你自己写节点。
> rclpy 是 ROS2 的官方 Python 客户端库——你未来把 RL 策略部署成节点时，写的就是它。

## 学习目标

1. 掌握一个 rclpy 节点的最小骨架：`init → Node → spin → shutdown`；
2. 会写 **Publisher / Subscription / Timer / Service**，理解回调驱动模型；
3. 会声明**参数**并监听参数变化；
4. 会用 Python **Launch 文件**一次启动多个节点并注入参数；
5. 理解 `spin` / Executor 的调度本质（单线程 vs 多线程）。

## ⚠️ 运行前提

本讲代码 cell 均为**可移植模板**（标注 `no-exec`，本机不执行）。
使用方法：在 ROS2 机器上 `ros2 pkg create` 建包后，把 cell 内容复制为 `.py` 文件运行。
仅最后的「Executor 概念模拟」为纯 Python，可真实执行。

## 1. 节点的最小骨架与回调驱动模型

**直觉**：rclpy 节点不是「从头跑到尾的脚本」，而是一个**注册了一堆回调的事件循环**。
`rclpy.spin(node)` 启动循环，此后程序完全由事件驱动：定时器到点、话题来消息、
服务被调用——对应回调被触发。这和你熟悉的 Gymnasium 环境 `step()` 循环**相反**：
控制权在框架手里（IoC，控制反转）。

类比：节点像一家餐厅——`__init__` 里挂好「点餐电话」（订阅）、「出菜广播」（发布）、
「定时巡台闹钟」（Timer），然后 `spin()` 开始营业，剩下就是等事件。

In [ ]:
# ⚠️ 运行前提：需要 ROS2 Jazzy 环境（Ubuntu 24.04 + apt 安装，见 docs/phase3_ros2.md）。
# 本机没有 ROS2，此 cell 仅作模板，保持未执行状态；请复制到 ROS2 工作区中运行。
"""minimal_talker.py —— 最小发布者（1 Hz 发布计数）。

在 ROS2 机器上：
    ros2 pkg create py_demos --build-type ament_python --dependencies rclpy std_msgs
把本文件放进 py_demos/py_demos/，在 setup.py 的 entry_points 注册：
    'console_scripts': ['talker = py_demos.minimal_talker:main']
然后 colcon build && ros2 run py_demos talker
"""
import rclpy
from rclpy.node import Node
from std_msgs.msg import String


class MinimalTalker(Node):
    def __init__(self):
        super().__init__("minimal_talker")          # 节点名（ros2 node list 中可见）
        self.pub = self.create_publisher(String, "chatter", 10)  # 类型/话题名/queue depth
        self.count = 0
        self.timer = self.create_timer(1.0, self.on_timer)       # 1 Hz 定时器

    def on_timer(self):
        msg = String()
        msg.data = f"hello {self.count}"
        self.pub.publish(msg)
        self.get_logger().info(f"发布: {msg.data}")
        self.count += 1


def main(args=None):
    rclpy.init(args=args)        # 初始化 DDS 上下文
    node = MinimalTalker()
    try:
        rclpy.spin(node)         # 事件循环：阻塞直到 Ctrl-C
    except KeyboardInterrupt:
        pass
    finally:
        node.destroy_node()
        rclpy.shutdown()


if __name__ == "__main__":
    main()

**预期输出**（`ros2 run py_demos talker`）：

```text
[INFO] [minimal_talker]: 发布: hello 0
[INFO] [minimal_talker]: 发布: hello 1
...
```

另开终端 `ros2 topic echo /chatter` 可看到消息流。
对应的订阅者只需把 `create_publisher` 换成：

```python
self.sub = self.create_subscription(String, "chatter", self.on_msg, 10)

def on_msg(self, msg):
    self.get_logger().info(f"收到: {msg.data}")
```

> 官方完整教程：[Writing a simple publisher and subscriber (Python)](https://docs.ros.org/en/jazzy/Tutorials/Beginner-Client-Libraries/Writing-A-Simple-Py-Publisher-And-Subscriber.html)。

## 2. 参数：节点的「旋钮」

参数让节点行为可调而不改代码。rclpy 中三步：**声明 → 读取 → （可选）监听变化**。
参数按节点作用域隔离，类型自动推断（`int/float/str/bool/数组`）。

In [ ]:
# ⚠️ 运行前提：需要 ROS2 Jazzy 环境（Ubuntu 24.04 + apt 安装，见 docs/phase3_ros2.md）。
# 本机没有 ROS2，此 cell 仅作模板，保持未执行状态；请复制到 ROS2 工作区中运行。
"""param_demo.py —— 声明参数、读取参数、监听运行时修改。

运行后在另一个终端试验：
    ros2 param list
    ros2 param get /param_demo publish_rate
    ros2 param set /param_demo publish_rate 5.0     # 观察回调日志
"""
import rclpy
from rcl_interfaces.msg import SetParametersResult
from rclpy.node import Node
from std_msgs.msg import String


class ParamDemo(Node):
    def __init__(self):
        super().__init__("param_demo")
        # 声明即注册：名称、默认值（类型由默认值推断）
        self.declare_parameter("publish_rate", 1.0)      # float
        self.declare_parameter("prefix", "tick")         # str
        self.declare_parameter("gains", [1.0, 0.1, 0.01])  # float 数组（如 PID）

        rate = self.get_parameter("publish_rate").value
        self.pub = self.create_publisher(String, "ticks", 10)
        self.timer = self.create_timer(1.0 / rate, self.on_timer)
        # 参数变化回调：ros2 param set 会触发它
        self.add_on_set_parameters_callback(self.on_param_change)
        self.count = 0

    def on_param_change(self, params):
        for p in params:
            if p.name == "publish_rate" and p.value <= 0:
                return SetParametersResult(successful=False, reason="rate 必须为正")
            self.get_logger().info(f"参数更新: {p.name} = {p.value}")
        return SetParametersResult(successful=True)

    def on_timer(self):
        prefix = self.get_parameter("prefix").value
        self.pub.publish(String(data=f"{prefix} {self.count}"))
        self.count += 1


def main(args=None):
    rclpy.init(args=args)
    node = ParamDemo()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        pass
    finally:
        node.destroy_node()
        rclpy.shutdown()

**预期输出**：`ros2 param set /param_demo publish_rate 5.0` 后，
节点日志打印 `参数更新: publish_rate = 5.0`（注意：定时器周期不会自动重建——
真正的工程做法是在回调里 `cancel` 旧 timer 再建新 timer，见练习 2）。

> RL 视角：参数 ≈ 环境的 `options`。部署策略节点时（W27），模型路径、控制频率、
> 观测话题名都应该做成参数，而不是写死在代码里。

## 3. Launch：一次拉起整个系统

真实系统动辄十几个节点。Launch 文件（Python 语法）描述「启动哪些节点、
各自什么参数、话题如何重映射」。对比手动开 10 个终端，launch 是可复现的系统编排。

In [ ]:
# ⚠️ 运行前提：需要 ROS2 Jazzy 环境（Ubuntu 24.04 + apt 安装，见 docs/phase3_ros2.md）。
# 本机没有 ROS2，此 cell 仅作模板，保持未执行状态；请复制到 ROS2 工作区中运行。
"""demo.launch.py —— 启动 talker + listener，并通过参数/重映射定制。

用法：放到包的 launch/ 目录，setup.py 的 data_files 注册后：
    ros2 launch py_demos demo.launch.py
或不经安装直接：ros2 launch <路径>/demo.launch.py
"""
from launch import LaunchDescription
from launch_ros.actions import Node


def generate_launch_description():
    return LaunchDescription([
        Node(
            package="py_demos",
            executable="talker",             # setup.py 里注册的入口名
            name="fast_talker",              # 覆盖节点名
            parameters=[{"publish_rate": 5.0, "prefix": "fast"}],  # 注入参数
            remappings=[("chatter", "fast_chatter")],              # 话题重映射
            output="screen",
        ),
        Node(
            package="py_demos",
            executable="listener",
            name="fast_listener",
            remappings=[("chatter", "fast_chatter")],  # 与 talker 对上前才能收到
            output="screen",
        ),
    ])

**预期输出**：两个节点的日志交错打印，`fast_talker` 以 5 Hz 发布到 `/fast_chatter`，
`fast_listener` 同步接收。`ros2 topic list` 中能看到 `/fast_chatter` 而没有 `/chatter`。

**重映射（remapping）是 ROS2 最重要的解耦手段**：节点代码里写「逻辑名」，
接线关系交给 launch——你的 RL 策略节点订阅 `/odom` 还是 `/robot/odom`，
部署时一行 launch 参数就能切换。

## 4. Executor：`spin` 到底在干什么

`rclpy.spin(node)` 等价于创建一个 **SingleThreadedExecutor**：一个循环不断地
「收集就绪事件 → 逐个执行回调」。推论：

- 默认**单线程**：一个回调阻塞（比如策略前向推理耗时 50 ms），同 executor 里
  其它回调（定时器、订阅）都会被饿死；
- `MultiThreadedExecutor` 用线程池并发执行回调，但你要自己处理共享数据的竞争；
- 高频控制节点常把「控制回调」和「慢速 IO」放进不同的 callback group。

下面的纯 Python 模拟（可在本机执行）复现 executor 的调度逻辑，
帮你直观看到「一个慢回调拖垮整个节点」：

In [1]:
"""纯 Python 模拟 SingleThreadedExecutor 的调度（只依赖标准库，本机真实执行）。"""
import heapq
import itertools
import time


class MiniExecutor:
    """极简事件循环：定时器到期 / 队列来消息时，按就绪时间顺序执行回调。"""

    def __init__(self):
        self._timers = []        # (下次触发时刻, 序号, 周期, 回调)
        self._queue = []         # (入队时刻, 序号, 消息, 回调)
        self._seq = itertools.count()
        self.lag_log = []        # (回调名, 实际触发 - 应当触发)

    def add_timer(self, period, callback, name):
        heapq.heappush(self._timers, [time.monotonic() + period, next(self._seq), period, callback, name])

    def feed(self, msg, callback, name):
        self._queue.append((time.monotonic(), next(self._seq), msg, callback, name))

    def spin_once(self):
        now = time.monotonic()
        # 先处理到期的定时器
        if self._timers and self._timers[0][0] <= now:
            due, seq, period, cb, name = heapq.heappop(self._timers)
            self.lag_log.append((name, now - due))
            cb()
            heapq.heappush(self._timers, [due + period, seq, period, cb, name])
        elif self._queue:
            t0, seq, msg, cb, name = self._queue.pop(0)
            self.lag_log.append((name, now - t0))
            cb(msg)
        else:
            time.sleep(0.001)


def demo():
    ex = MiniExecutor()
    # 控制定时器：要求 10 ms 周期（类似 100 Hz 控制回路）
    ex.add_timer(0.010, lambda: None, "control_100Hz")
    # 一个「慢」回调：模拟每次花 40 ms 的神经网络推理
    def slow_policy(msg):
        time.sleep(0.040)
    for _ in range(10):
        ex.feed("obs", slow_policy, "policy_inference")

    t_end = time.monotonic() + 0.5
    while time.monotonic() < t_end:
        ex.spin_once()

    lags = [lag * 1000 for name, lag in ex.lag_log if name == "control_100Hz"]
    print(f"控制回调次数: {len(lags)}, 平均延迟 {sum(lags)/len(lags):.1f} ms, "
          f"最大延迟 {max(lags):.1f} ms（周期只有 10 ms！）")


demo()

控制回调次数: 49, 平均延迟 12.8 ms, 最大延迟 31.0 ms（周期只有 10 ms！）


**预期输出**：控制回调平均延迟远大于 10 ms 周期（因为慢回调 40 ms 会阻塞循环）。
结论：部署 RL 策略时，**要么把推理优化到远小于控制周期，要么用多线程/callback group 隔离**。

## ✏️ 练习

### 练习 1（★★，约 30 分钟）：温度监控对

写一对节点 `temp_sensor`（以 2 Hz 发布随机温度 `std_msgs/Float64`，均值 25、噪声 σ=1）
和 `temp_monitor`（订阅温度，超过 28 时 `get_logger().warn` 报警，并统计最近 10 条的均值）。
交付：两个 `.py` 文件 + 运行截图/日志文本 + `ros2 topic hz /temperature` 输出。

### 练习 2（★★，约 30 分钟）：真正可调速的 talker

扩展本讲的 `ParamDemo`：在参数回调中**销毁并重建定时器**，使
`ros2 param set /param_demo publish_rate X` 立即生效；对非法值（≤0 或 >1000）拒绝并给出理由。
交付：代码 + 一段 `ros2 param set` 前后 `ros2 topic hz` 变化的对比输出。

### 练习 3（★★，约 25 分钟）：三节点 launch

写一个 launch 文件同时启动：talker（重映射到 `/ch1`）、talker（重映射到 `/ch2`）、
listener（通过参数选择订阅 `/ch1` 或 `/ch2`，参数名 `channel`）。
要求 `ros2 launch py_demos three.launch.py channel:=/ch2` 可切换。
交付：launch 文件 + 两种启动方式下 `ros2 topic info` 的输出差异。

### 练习 4（★★★，约 45 分钟）：服务 + 异步客户端

写一个 `AddTwoInts` 服务端（`example_interfaces/srv/AddTwoInts`），以及一个
**在 spin 中定时发起异步调用**的客户端（提示：`self.cli.call_async(req)` 返回 Future，
用 `add_done_callback` 处理结果；禁止在回调里 `spin_until_future_complete` 嵌套 spin）。
交付：代码 + 日志；并用一段话解释为什么回调内嵌套 spin 是危险的（结合第 4 节的 Executor 模型）。

## 参考答案

<details>
<summary>参考答案</summary>

**练习 1**（核心片段）：

```python
class TempSensor(Node):
    def __init__(self):
        super().__init__("temp_sensor")
        self.pub = self.create_publisher(Float64, "temperature", 10)
        self.create_timer(0.5, self.on_timer)

    def on_timer(self):
        self.pub.publish(Float64(data=random.gauss(25.0, 1.0)))

class TempMonitor(Node):
    def __init__(self):
        super().__init__("temp_monitor")
        self.buf = deque(maxlen=10)
        self.create_subscription(Float64, "temperature", self.on_msg, 10)

    def on_msg(self, msg):
        if msg.data > 28.0:
            self.get_logger().warn(f"高温报警: {msg.data:.1f}")
        self.buf.append(msg.data)
        self.get_logger().info(f"均值: {sum(self.buf)/len(self.buf):.2f}")
```

**练习 2**：在 `on_param_change` 中检测到 `publish_rate` 更新后：

```python
self.timer.cancel()
self.timer = self.create_timer(1.0 / new_rate, self.on_timer)
```

校验：`if not (0 < p.value <= 1000): return SetParametersResult(successful=False, reason="rate 超出 (0,1000]")`。

**练习 3**：listener 声明 `channel` 参数，`create_subscription(String, self.get_parameter("channel").value, ...)`；
launch 中用 `LaunchConfiguration("channel")` + `Node(parameters=[{"channel": LaunchConfiguration("channel")}])`。

**练习 4**（客户端核心）：

```python
def on_timer(self):
    if self.future is not None and not self.future.done():
        return  # 上一次请求未完成，避免堆积
    req = AddTwoInts.Request(a=random.randint(0, 9), b=random.randint(0, 9))
    self.future = self.cli.call_async(req)
    self.future.add_done_callback(self.on_response)

def on_response(self, future):
    self.get_logger().info(f"sum = {future.result().sum}")
```

嵌套 spin 危险的原因：回调在 executor 线程中执行，`spin_until_future_complete`
会再次驱动同一 executor 处理事件——回调重入、状态交错，单线程 executor 下直接死锁，
多线程下产生难以复现的竞争。
</details>

## 延伸阅读

- [官方教程：Writing a simple publisher and subscriber (Python)](https://docs.ros.org/en/jazzy/Tutorials/Beginner-Client-Libraries/Writing-A-Simple-Py-Publisher-And-Subscriber.html)
- [官方教程：Writing a simple service and client (Python)](https://docs.ros.org/en/jazzy/Tutorials/Beginner-Client-Libraries/Writing-A-Simple-Py-Service-And-Client.html)
- [官方教程：Launch 文件](https://docs.ros.org/en/jazzy/Tutorials/Intermediate/Launch/Launch-Main.html)
- [rclpy 示例仓库 ros2/examples](https://github.com/ros2/examples)（最小节点大全，适合精读）
- 下一讲预告：W23 用 TF2 管理坐标变换，并手写第一份 URDF 机器人模型。